In [ ]:
# Codigo para encontrar usuarios comunes en archivos de predicciones
import pandas as pd
import os
import sys
import json
import glob
from functools import reduce
import numpy as np

# --- Celda 1: Funciones Helper ---

# (Reutilizamos la función de guardado de run_experiment_saves.py)
def save_json_with_numpy(data, filepath):
    """Guarda datos (incluyendo tipos numpy) como JSON."""
    
    class NumpyEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, (np.integer, np.int64)):
                return int(obj)
            elif isinstance(obj, (np.floating, np.float32, np.float64)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            return super(NumpyEncoder, self).default(obj)

    def convert_keys_to_string(obj):
        if isinstance(obj, dict):
            return {str(k) if isinstance(k, (int, np.integer, np.int64)) else k: convert_keys_to_string(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_keys_to_string(i) for i in obj]
        return obj

    # Asegurarse de que los datos (como un set) se conviertan a lista primero
    if isinstance(data, set):
        data = list(data)
        
    data_to_save = convert_keys_to_string(data)
    with open(filepath, 'w') as f:
        json.dump(data_to_save, f, indent=4, cls=NumpyEncoder)

def find_common_users(data_path):
    """
    Encuentra la intersección de usuarios que tienen predicciones
    en TODOS los archivos *_predictions.csv dentro de un directorio.
    """
    print(f"Buscando usuarios comunes en: {data_path}")
    
    # Encontrar todos los archivos de predicciones
    pred_files = glob.glob(os.path.join(data_path, "*_predictions.csv"))
    
    if not pred_files:
        print(f"  [ERROR] No se encontraron archivos *_predictions.csv en {data_path}.")
        print("  Asegúrate de haber ejecutado run_experiment_saves.py primero.")
        return None # Devolver None si no se encuentran archivos

    all_user_sets = []
    
    # Cargar los userIDs de cada archivo
    for f in pred_files:
        model_name = os.path.basename(f).replace("_predictions.csv", "")
        try:
            df = pd.read_csv(f)
            # Asegurarse de que la columna exista y no esté vacía
            if 'userId' in df.columns and not df.empty:
                # Convertir userId a int para consistencia (importante si algunos son float)
                user_set = set(df['userId'].dropna().astype(int).unique())
                all_user_sets.append(user_set)
                print(f"  Encontrados {len(user_set)} usuarios únicos para el modelo: {model_name}")
            else:
                print(f"  Advertencia: Archivo {f} está vacío o no tiene columna 'userId'.")
        except pd.errors.EmptyDataError:
            print(f"  Advertencia: Archivo {f} está vacío.")
        except Exception as e:
            print(f"  [ERROR] No se pudo leer {f}: {e}")

    if not all_user_sets:
        print("  [ERROR] No se pudieron cargar conjuntos de usuarios válidos.")
        return None

    # Encontrar la intersección (usuarios comunes a todos)
    common_users = reduce(lambda a, b: a.intersection(b), all_user_sets)
    
    if not common_users:
        print("  [ERROR] No se encontró ningún usuario común en todos los archivos de predicción.")
        return None

    print(f"\nSe encontraron {len(common_users)} usuarios comunes en todos los modelos.")
    
    # Guardar la lista de usuarios comunes
    output_file = os.path.join(data_path, 'common_users.json')
    try:
        # Convertir set a lista para guardarlo en JSON
        save_json_with_numpy(list(common_users), output_file)
        print(f"Lista de usuarios comunes guardada en: {output_file}")
    except Exception as e:
        print(f"  [ERROR] No se pudo guardar el archivo JSON: {e}")
        
    return common_users

for DATASET_PERCENTAGE in ['10', '25', '50', '75', '100']:
    #%%
    # --- Celda 2: Ejecución ---

    # Definir la ruta basada en la configuración
    data_path = os.path.join('data', DATASET_PERCENTAGE)

    if not os.path.exists(data_path):
        print(f"[ERROR] El directorio de datos no existe: {data_path}")
    else:
        # Llamar a la función
        common_users_set = find_common_users(data_path)
        if common_users_set:
            print(f"\nAnálisis completado. {len(common_users_set)} usuarios comunes encontrados.")
        else:
            print("\nAnálisis completado. No se encontraron usuarios comunes.")

In [13]:
import pandas as pd
import numpy as np
import json
import os
import sys
import glob
from functools import reduce
import subprocess # <<<<<<< NUEVA IMPORTACIÓN

# --- Configuración ---
BASE_PATH = 'data'
PERCENTAGES = ['10', '25', '50', '75', '100']
# Usaremos cuantiles para definir las categorías (ej. 20% inferior/superior)
QUANTILE_THRESHOLD = 0.20 

print(f"Iniciando análisis de perfiles persistentes en todos los datasets: {PERCENTAGES}")

#%%
# --- Celda 1: Cargar todos los datos y calcular estadísticas ---
# (Esta celda permanece sin cambios)

all_stats = {} # Diccionario para guardar los DataFrames de estadísticas
all_common_users = {} # Diccionario para guardar los sets de usuarios comunes

print("Cargando y procesando estadísticas para cada subconjunto...")

for perc in PERCENTAGES:
    print(f"\n--- Procesando {perc}% ---")
    DATASET_DIR = os.path.join(BASE_PATH, perc)
    TRAIN_FILE = os.path.join(DATASET_DIR, 'train.csv')
    COMMON_USERS_FILE = os.path.join(DATASET_DIR, 'common_users.json')
    
    try:
        # Cargar train.csv y common_users.json
        train_df = pd.read_csv(TRAIN_FILE)
        with open(COMMON_USERS_FILE, 'r') as f:
            common_user_ids = json.load(f)
        
        common_users_set = set(common_user_ids)
        all_common_users[perc] = common_users_set
        
        print(f"Cargados {len(train_df)} ratings y {len(common_users_set)} usuarios comunes.")
        
        # Calcular métricas de comportamiento (igual que antes)
        user_activity = train_df['userId'].value_counts().to_frame(name='rating_count')
        user_rating_behavior = train_df.groupby('userId')['rating'].agg(['mean', 'std']).rename(columns={'mean': 'rating_avg', 'std': 'rating_std'}).fillna(0)
        
        item_popularity = train_df['movieId'].value_counts().to_dict()
        train_df_with_pop = train_df.copy()
        train_df_with_pop['item_popularity'] = train_df_with_pop['movieId'].map(item_popularity)
        user_niche_score = train_df_with_pop.groupby('userId')['item_popularity'].mean().to_frame(name='avg_item_popularity')
        
        # Combinar y guardar solo para usuarios comunes
        stats_df = pd.concat([user_activity, user_rating_behavior, user_niche_score], axis=1)
        # Asegurarse de que el índice de stats_df sea del mismo tipo que common_users_set (int)
        stats_df.index = stats_df.index.astype(int)
        common_stats_df = stats_df[stats_df.index.isin(common_users_set)].copy()
        
        all_stats[perc] = common_stats_df
        print(f"Estadísticas para {len(common_stats_df)} usuarios comunes calculadas.")
        
    except FileNotFoundError as e:
        print(f"[ERROR] No se encontró un archivo para el {perc}%: {e}")
        print("Asegúrate de haber ejecutado 'run_all_experiments.sh' y 'find_common_users.py' para TODOS los datasets.")
        # Romper el bucle si falta un archivo
        break
    except Exception as e:
        print(f"[ERROR] Ocurrió un error inesperado procesando el {perc}%: {e}")
        break

print("\n--- Carga de datos completada ---")

#%%
# --- Celda 2: Encontrar los Usuarios Persistentes ---
# (Esta celda permanece sin cambios)

if len(all_stats) == len(PERCENTAGES): # Solo si se cargaron todos
    print(f"Calculando perfiles persistentes usando un umbral de cuantil de {QUANTILE_THRESHOLD*100}%...")
    
    # Inicializar sets con los usuarios del primer dataset (10%)
    if '10' not in all_stats:
        print("[ERROR] No se pueden inicializar los sets, faltan datos del 10%.")
    else:
        # Definir sets vacíos primero para asegurar que existan
        persistent_power_users = set()
        persistent_cold_users = set()
        persistent_critics = set()
        persistent_fans = set()
        persistent_niche = set()
        persistent_mainstream = set()

        q_low_act = all_stats['10']['rating_count'].quantile(QUANTILE_THRESHOLD)
        q_high_act = all_stats['10']['rating_count'].quantile(1 - QUANTILE_THRESHOLD)
        q_low_avg = all_stats['10']['rating_avg'].quantile(QUANTILE_THRESHOLD)
        q_high_avg = all_stats['10']['rating_avg'].quantile(1 - QUANTILE_THRESHOLD)
        q_low_pop = all_stats['10']['avg_item_popularity'].quantile(QUANTILE_THRESHOLD)
        q_high_pop = all_stats['10']['avg_item_popularity'].quantile(1 - QUANTILE_THRESHOLD)
        
        # Inicializar los sets persistentes
        persistent_power_users = set(all_stats['10'][all_stats['10']['rating_count'] >= q_high_act].index)
        persistent_cold_users = set(all_stats['10'][all_stats['10']['rating_count'] <= q_low_act].index)
        persistent_critics = set(all_stats['10'][all_stats['10']['rating_avg'] <= q_low_avg].index)
        persistent_fans = set(all_stats['10'][all_stats['10']['rating_avg'] >= q_high_avg].index)
        persistent_niche = set(all_stats['10'][all_stats['10']['avg_item_popularity'] <= q_low_pop].index)
        persistent_mainstream = set(all_stats['10'][all_stats['10']['avg_item_popularity'] >= q_high_pop].index)
        
        # Iterar por el resto (25% a 100%) y encontrar la intersección
        for perc in PERCENTAGES[1:]:
            print(f"Calculando intersección con el dataset {perc}%...")
            stats_df = all_stats[perc]
            
            # Calcular cuantiles para ESTE dataset
            q_low_act = stats_df['rating_count'].quantile(QUANTILE_THRESHOLD)
            q_high_act = stats_df['rating_count'].quantile(1 - QUANTILE_THRESHOLD)
            q_low_avg = stats_df['rating_avg'].quantile(QUANTILE_THRESHOLD)
            q_high_avg = stats_df['rating_avg'].quantile(1 - QUANTILE_THRESHOLD)
            q_low_pop = stats_df['avg_item_popularity'].quantile(QUANTILE_THRESHOLD)
            q_high_pop = stats_df['avg_item_popularity'].quantile(1 - QUANTILE_THRESHOLD)
            
            # Actualizar los sets persistentes con la intersección
            persistent_power_users.intersection_update(set(stats_df[stats_df['rating_count'] >= q_high_act].index))
            persistent_cold_users.intersection_update(set(stats_df[stats_df['rating_count'] <= q_low_act].index))
            persistent_critics.intersection_update(set(stats_df[stats_df['rating_avg'] <= q_low_avg].index))
            persistent_fans.intersection_update(set(stats_df[stats_df['rating_avg'] >= q_high_avg].index))
            persistent_niche.intersection_update(set(stats_df[stats_df['avg_item_popularity'] <= q_low_pop].index))
            persistent_mainstream.intersection_update(set(stats_df[stats_df['avg_item_popularity'] >= q_high_pop].index))

        print("\n--- Búsqueda de Perfiles Persistentes Completada ---")
        print(f"Se encontraron {len(persistent_power_users)} 'Power Users' persistentes.")
        print(f"Se encontraron {len(persistent_cold_users)} 'Cold-Start Users' persistentes.")
        print(f"Se encontraron {len(persistent_critics)} 'Critics' persistentes.")
        print(f"Se encontraron {len(persistent_fans)} 'Fans' persistentes.")
        print(f"Se encontraron {len(persistent_niche)} 'Niche Seekers' persistentes.")
        print(f"Se encontraron {len(persistent_mainstream)} 'Mainstream Users' persistentes.")
else:
    print("El análisis no se puede completar porque no se cargaron todos los datasets.")

#%%
# --- Celda 3: Seleccionar el Usuario Más Representativo de cada Categoría ---
# (Basado en el mayor/menor rating_count del dataset 100%)

print("\n--- Seleccionando el Usuario MÁS Representativo de cada categoría ---")
print("(El usuario persistente con el recuento de ratings más relevante en el dataset del 100%)")

# <<<<<<< LÓGICA CORREGIDA Y SIMPLIFICADA >>>>>>>
if 'all_stats' in locals() and '100' in all_stats and not all_stats['100'].empty:
    
    stats_100 = all_stats['100'] # Asignar el DataFrame
    representative_users = {} # Inicializar el diccionario de resultados

    def find_most_representative(user_ids_set, category_name, description, sort_ascending=False):
        """
        Encuentra el usuario más representativo de un set,
        ordenando por rating_count.
        """
        print(f"\n--- {category_name} ---")
        print(f"Descripción: {description}")
        
        if not user_ids_set:
            print("No se encontraron usuarios persistentes para esta categoría.")
            return None
        
        candidates_df = stats_100[stats_100.index.isin(user_ids_set)].sort_values(by='rating_count', ascending=sort_ascending)
        
        if candidates_df.empty:
            print("No se encontraron estadísticas para los usuarios persistentes de esta categoría.")
            return None
        
        print("Top 5 candidatos más representativos (basado en stats del 100%):")
        print(candidates_df.head(5))
        
        best_user_id = candidates_df.index[0]
        representative_users[category_name] = best_user_id
        return best_user_id

    # Encontrar el mejor para cada categoría
    if 'persistent_power_users' in locals():
        find_most_representative(
            persistent_power_users, 
            "Power User",
            "Usuarios que están consistentemente en el 20% superior de actividad. Se selecciona el más activo.",
            sort_ascending=False
        ) 
        find_most_representative(
            persistent_cold_users, 
            "Cold-Start User",
            "Usuarios que están consistentemente en el 20% inferior de actividad. Se selecciona el menos activo.",
            sort_ascending=True
        ) 
        find_most_representative(
            persistent_critics, 
            "Critic",
            "Usuarios que están consistentemente en el 20% inferior de rating promedio. Se selecciona el más activo de este grupo.",
            sort_ascending=False
        ) 
        find_most_representative(
            persistent_fans, 
            "Fan",
            "Usuarios que están consistentemente en el 20% superior de rating promedio. Se selecciona el más activo de este grupo.",
            sort_ascending=False
        ) 
        find_most_representative(
            persistent_niche, 
            "Niche Seeker",
            "Usuarios que consistentemente prefieren ítems de baja popularidad (long-tail). Se selecciona el más activo de este grupo.",
            sort_ascending=False
        ) 
        find_most_representative(
            persistent_mainstream, 
            "Mainstream User",
            "Usuarios que consistentemente prefieren ítems de alta popularidad (blockbusters). Se selecciona el más activo de este grupo.",
            sort_ascending=False
        ) 

        print("\n\n--- RESUMEN DE USUARIOS RECOMENDADOS PARA PRUEBAS ---")
        if representative_users:
            for category, user_id in representative_users.items():
                print(f"El mejor '{category}':\t userId = {user_id}")
            print("\nIniciando la ejecución de 'get_top10_recommendations.py' para estos usuarios...")
        else:
            print("No se seleccionaron usuarios representativos.")
            
    else:
        print("Error: Las variables 'persistent_...' no se encontraron. Asegúrate de ejecutar la Celda 2.")
else:
    print("No se pudo completar la selección de usuarios representativos.")
    print("Causa probable: Las estadísticas del 100% ('all_stats['100']') no se cargaron o están vacías.")
    print("Asegúrate de que 'run_all_experiments.sh' y 'find_common_users.py' se hayan ejecutado para el 100% del dataset.")

Iniciando análisis de perfiles persistentes en todos los datasets: ['10', '25', '50', '75', '100']
Cargando y procesando estadísticas para cada subconjunto...

--- Procesando 10% ---
Cargados 80016 ratings y 3249 usuarios comunes.
Estadísticas para 3249 usuarios comunes calculadas.

--- Procesando 25% ---
Cargados 200041 ratings y 3466 usuarios comunes.
Estadísticas para 3466 usuarios comunes calculadas.

--- Procesando 50% ---
Cargados 400083 ratings y 3579 usuarios comunes.
Estadísticas para 3579 usuarios comunes calculadas.

--- Procesando 75% ---
Cargados 600124 ratings y 3638 usuarios comunes.
Estadísticas para 3638 usuarios comunes calculadas.

--- Procesando 100% ---
Cargados 800167 ratings y 3675 usuarios comunes.
Estadísticas para 3675 usuarios comunes calculadas.

--- Carga de datos completada ---
Calculando perfiles persistentes usando un umbral de cuantil de 20.0%...
Calculando intersección con el dataset 25%...
Calculando intersección con el dataset 50%...
Calculando inter